In [1]:
import sys
import pandas as pd
import logging
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import database_interact as dbi
import pubchem_retrieval as pcr
import load_tools as ldt
import lcmsruns_tools as lrt
import ms1_ms2_analysis as msa
import rt_align_tools as rat
import targeted_analysis as tga
import targeted_gui as tgi
import logging_config as lcf
#import spectrum_handlers as sph

logger = lcf.setup_logging(log_level=logging.INFO)
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
ANALYSIS_OUTPUT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}'

In [3]:
new_main_database = False
new_atlases = False
new_rt_alignment = False
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-7906f4bf395d4ed791f3763ed10f61c8"
    TARGET_ATLAS_UID = "atl-raw-2e24a9969a6a4758844b99162e9e0a03"
    logger.info(f"Skipping new atlas creation and using QC Atlas: {QC_ATLAS_UID} and TARGET Atlas: {TARGET_ATLAS_UID}")

2025-09-05 10:42:57 | metatlas2 | INFO | Skipping new atlas creation and using QC Atlas: atl-raw-7906f4bf395d4ed791f3763ed10f61c8 and TARGET Atlas: atl-raw-2e24a9969a6a4758844b99162e9e0a03


In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-0c89e582233e481ca4f1920611f6007f"
    logger.info(f"Skipping new RT alignment and using {ANALYSIS_ATLAS_UID}")

2025-09-05 10:42:57 | metatlas2 | INFO | Skipping new RT alignment and using atl-rta-0c89e582233e481ca4f1920611f6007f


In [7]:
if new_analysis is True:
    # Run targeted analysis workflow - now returns ProjectAnalysis object
    atlas_dataframe, project_analysis = tga.run_targeted_analysis_workflow(
        PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG
    )

    logger.info(f"Analysis complete:")
    logger.info(f"  Total compounds: {len(project_analysis.compounds)}")

    # Get analysis summary
    summary = project_analysis.get_analysis_summary()
    logger.info(f"  Compounds with EIC data: {summary['compounds_with_eic']}")
    logger.info(f"  Compounds with MS2 data: {summary['compounds_with_ms2']}")

    # Create GUI
    gui = tgi.create_gui(project_analysis, CONFIG, PROJECT_DB_PATH)
    gui._project_analysis = project_analysis  # Store ProjectAnalysis object for new workflow
    display(gui)

2025-09-05 10:42:57 | metatlas2.targeted_analysis | INFO | Setting up targeted analysis database...
2025-09-05 10:42:57 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-05 10:42:57 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-05 10:42:57 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-05 10:42:57 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-05 10:42:57 | metatlas2.database_interact | INFO | Enriched 65 compounds with metadata from main database
2025-09-05 10:42:57 | metatlas2.targeted_analysis | INFO | Created Atlas dataframe with 65 compounds
2025-09-05 10:42:57 | metatlas2.simple_cache | INFO | Progress checkpoint saved: initialized at 20250905_104257
2025-09-05 10:42:57 | metatlas2.targeted_analysis | INFO | Loading experimental files

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | WARNING |   Completed 1/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5 - No data extracted
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | WARNING |   Completed 2/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5 - No data extracted
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | WARNING |   Completed 3/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5 - No data extracted
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | WARNING |   Completed 4/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5 - No data extracted
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | WARNING |   Co

Worker 0: Error processing 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5: 'atlas'
Worker 0: Traceback: Traceback (most recent call last):
  File "/Users/BKieft/Metabolomics/metatlas2/metatlas2/ms1_ms2_analysis.py", line 697, in _process_single_file_simple
    data = ftt.get_data(file_input, save_file=False, return_data=True, ms1_feature_filter=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/BKieft/Metabolomics/metatlas2/metatlas2/feature_tools.py", line 447, in get_data
    d = get_atlas_data_from_file(input_data['lcmsrun'],input_data['atlas'],desired_key='ms1_%s'%polarity_short_string)
                                                       ~~~~~~~~~~^^^^^^^^^
KeyError: 'atlas'

Worker 2: Error processing 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1

2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | INFO | Parallel processing complete: 0 compounds with data
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | INFO | Extraction complete:
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | INFO |   Total compounds with data: 0
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with EIC data: 0
2025-09-05 10:43:01 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 data: 0
2025-09-05 10:43:01 | metatlas2.simple_cache | INFO | Progress checkpoint saved: data_extracted at 20250905_104301
2025-09-05 10:43:01 | metatlas2.simple_cache | INFO | Analysis cache saved: /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/cache/analysis/atl-rta-0c89e582233e481ca4f1920611f6007f/project_analysis_20250905_104301.pkl
2025-09-05 10:43:01 | metatlas2.simple_cache | INFO |   Compounds: 65
2025-09-05 10:43:01 | metatlas2.simple_cache | INFO |   EIC data: 0


AttributeError: 'NoneType' object has no attribute '_project_analysis'

In [ ]:
if 'project_analysis' in locals():
    sample_inchi_key = 'AGPKZVBTJJNPAG-WHFBIAKZSA-N'
    sample_compound = project_analysis.compounds[sample_inchi_key]
    print(sample_compound)
    print(f"Sample compound: {sample_compound.compound_name}")
    print(f"Original RT: {sample_compound.original_rt_peak}")
    print(f"Current RT: {sample_compound.rt_peak}")
    print(f"Is modified: {sample_compound.is_rt_modified or sample_compound.is_annotation_modified}")
    print(f"EIC files: {len(sample_compound.eic_data_files)}")
    print(f"MS2 files: {len(sample_compound.ms2_data_files.get('files', {}))}")

In [ ]:
if new_analysis is True:
    post_analysis_atlas_uid, targeted_analysis_uid, comprehensive_report = tga.run_post_analysis_workflow_v2(
        project_db_path=PROJECT_DB_PATH,
        analysis_atlas_uid=ANALYSIS_ATLAS_UID,
        project_analysis=gui._project_analysis,  # Use ProjectAnalysis object
        atlas_dataframe=atlas_dataframe,
        project_name=PROJECT_NAME,
        config=CONFIG,
        analysis_output_path=ANALYSIS_OUTPUT_PATH
    )
    
    logger.info("Class-based workflow completed successfully!")

# Print results
logger.info(f"Analysis saved with UID: {targeted_analysis_uid}")
logger.info(f"Post-analysis atlas UID: {post_analysis_atlas_uid}")
logger.info(f"Report saved to: {ANALYSIS_OUTPUT_PATH}")

# Show cache status
project_dir = str(Path(PROJECT_DB_PATH).parent)
cache_status = scache.get_cache_status(project_dir, ANALYSIS_ATLAS_UID)
logger.info(f"Cache status: {cache_status['total_caches']} caches, {cache_status['total_checkpoints']} checkpoints")

In [ ]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)